# 01 - Recoleccion de datos

Este notebook localiza el archivo original, lo copia a `data/raw/` sin modificarlo y
revisa la calidad basica de los datos (nulos, duplicados, tipos y rangos).


## Constantes del proyecto


In [15]:
from pathlib import Path
import shutil

# Variable objetivo (continua) y predictores elegidos a partir del archivo
TARGET = 'produccion_toneladas'
FEATURES = ['año', 'id_municipio', 'subregion']
CULTIVO = 'Plátano'
SEED = 42

# Localizar la raiz del proyecto (subir un nivel si se ejecuta desde notebooks/)
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_RAW = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
FIGURES = ROOT / 'figures'

for carpeta in (DATA_RAW, DATA_PROCESSED, FIGURES):
    carpeta.mkdir(parents=True, exist_ok=True)

ARCHIVO_FUENTE = ROOT / 'files' / 'Producción_de_frutales_en_el_Valle_del_Cauca_20260919.csv'
ARCHIVO_RAW = DATA_RAW / 'Producción_de_frutales_en_el_Valle_del_Cauca_20260919.csv'

print('TARGET   =', TARGET)
print('FEATURES =', FEATURES)
print('CULTIVO  =', CULTIVO)
print('SEED     =', SEED)


TARGET   = produccion_toneladas
FEATURES = ['año', 'id_municipio', 'subregion']
CULTIVO  = Plátano
SEED     = 42


## ¿Por que estas constantes?

- **Objetivo (`TARGET = 'produccion_toneladas'`)**: es la unica variable numerica continua
  (toneladas producidas) que queremos predecir.
- **`año`**: tendencia temporal de la produccion (de 2000 a 2022).
- **`id_municipio`**: identificador numerico del municipio (ubicacion).
- **`subregion`**: region geografica del Valle del Cauca (4 categorias, categorica).

Notas:

- El dataset cubre **38 cultivos**; para no mezclar especies muy distintas en un mismo modelo,
  se filtra solo el cultivo con mas datos (**Plátano**, 943 filas). La seleccion se hace en el
  notebook 02 usando la constante `CULTIVO`.
- **`ciclo` se descarta**: para Plátano es siempre `Anual` (constante, no aporta informacion).
- **`subregion` es categorica**; en el notebook 03 se codifica con `pd.get_dummies(drop_first=True)`.
- Se descartan `municipio` (texto, duplicado del id) y `cultivo`/`id_cultivo` (constantes tras filtrar).


## Copia del archivo original a data/raw/


In [16]:
shutil.copy2(ARCHIVO_FUENTE, ARCHIVO_RAW)
print('Archivo copiado a:', ARCHIVO_RAW)


Archivo copiado a: D:\Universidad\Alejandro\Taller_001\data\raw\Producción_de_frutales_en_el_Valle_del_Cauca_20260919.csv


## Tabla de trazabilidad

| Campo | Valor |
| --- | --- |
| Fuente | Registros de produccion de frutales en el Valle del Cauca (Colombia) |
| URL | https://www.datos.gov.co/browse?q=venta+&sortBy=relevance&pageSize=20 |
| Fecha de obtencion | 2026-09-19 |
| Formato | CSV (campos entre comillas; decimal con coma) |
| Filas | 9 992 |
| Columnas | 8 |


## Pasos de recoleccion

1. Se identifico el archivo fuente en `./files/`.
2. Se copio (sin modificar) a `data/raw/` con `shutil.copy2`.
3. Se leyo con `pandas` usando separador por comas y comillas (`sep=','`).
4. Se reviso la estructura: nombres de columna, forma del dataset y primeras filas.
5. Se aplicaron criterios de calidad (nulos, duplicados, tipos y rangos).


In [17]:
import pandas as pd

df = pd.read_csv(ARCHIVO_RAW, sep=',', encoding='utf-8')
print('Forma (filas, columnas):', df.shape)
print('Columnas:', df.columns.tolist())
df.head()


Forma (filas, columnas): (9992, 8)
Columnas: ['año', 'Id_municipio', 'Municipio', 'Subregion', 'Id_cultivo', 'Cultivo', 'Ciclo', 'Produccion_toneladas']


,año,Id_municipio,Municipio,Subregion,Id_cultivo,Cultivo,Ciclo,Produccion_toneladas
0,2022,76233,Dagua,Sur,2045801,Plátano,Anual,14.088
1,2022,76100,Bolivar,Norte,2045801,Plátano,Anual,6.578
2,2022,76243,El Aguila,Norte,2045801,Plátano,Anual,26.000
3,2022,76001,Cali,Sur,2045801,Plátano,Anual,186
4,2022,76275,Florida,Sur,2045801,Plátano,Anual,"2.124,4"


## Criterios de calidad revisados


In [18]:
# 1) Valores nulos
print('Nulos por columna:')
print(df.isnull().sum())


Nulos por columna:
año                     0
Id_municipio            0
Municipio               0
Subregion               0
Id_cultivo              0
Cultivo                 0
Ciclo                   0
Produccion_toneladas    0
dtype: int64


In [19]:
# 2) Filas duplicadas
print('Filas duplicadas:', df.duplicated().sum())


Filas duplicadas: 39


In [20]:
# 3) Tipos de dato al leer el CSV
print('Tipos de dato al leer el CSV:')
print(df.dtypes)


Tipos de dato al leer el CSV:
año                      int64
Id_municipio             int64
Municipio               object
Subregion               object
Id_cultivo               int64
Cultivo                 object
Ciclo                   object
Produccion_toneladas    object
dtype: object


In [21]:
# 4) Rangos razonables

# Rango del año
print('año -> min:', df['año'].min(), '| max:', df['año'].max())
print('año -> valores unicos:', sorted(df['año'].unique().tolist()))

# Ejemplos de produccion en bruto (formato español)
print('\nEjemplos de Produccion_toneladas (cruda):',
      df['Produccion_toneladas'].head(10).tolist())

# Variables categoricas
print('\nSubregion -> valores unicos:', df['Subregion'].unique().tolist())
print('Ciclo -> valores unicos:', df['Ciclo'].unique().tolist())
print('Cultivo -> numero de categorias:', df['Cultivo'].nunique())
print('Municipio -> numero de categorias:', df['Municipio'].nunique())

# Identificadores numericos
print('\nId_municipio es numerico:', df['Id_municipio'].dtype.kind in 'iu')
print('Id_cultivo es numerico:', df['Id_cultivo'].dtype.kind in 'iu')

# Ceros y negativos en la produccion
prod = (df['Produccion_toneladas']
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .astype(float))
print('\nProduccion_toneladas = 0:', (prod == 0).sum())
print('Produccion_toneladas negativas:', (prod < 0).sum())


año -> min: 2000 | max: 2022
año -> valores unicos: [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]

Ejemplos de Produccion_toneladas (cruda): ['14.088', '6.578', '26.000', '186', '2.124,4', '416', '1.110', '12.000', '1.957', '1.176']

Subregion -> valores unicos: ['Sur', 'Norte', 'Centro', 'Pacifico']
Ciclo -> valores unicos: ['Anual', 'Semestre 1', 'Semestre 2']
Cultivo -> numero de categorias: 38
Municipio -> numero de categorias: 42

Id_municipio es numerico: True
Id_cultivo es numerico: True

Produccion_toneladas = 0: 202
Produccion_toneladas negativas: 0


## Resumen de calidad

- **Nulos**: ninguno en todo el dataset.
- **Duplicados**: 39 filas duplicadas (se eliminan en el notebook 02).
- **Tipos**: `año`, `id_municipio` e `id_cultivo` se leen como enteros; la produccion queda como texto y debe convertirse a numero.
- **Rangos**: el año va de 2000 a 2022; la produccion usa formato español (punto=miles, coma=decimal)
  y contiene 202 ceros; `subregion` tiene 4 categorias, `ciclo` 3 y `cultivo` 38.
